In [1]:
import scipy.optimize as optimize
import numpy as np
import zfit
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING
from PDFs import *


/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress the TensorFlow warnings AND this warning, set ZFIT_DISABLE_TF_WARNINGS=1.
  warnings.warn(


In [2]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")


recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=22)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=2)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=2)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [3]:

def save_phy_results_slsqp(result_scipy, phys_dict, cov_phys, obs_order, bin_n, base_dir="fit_results_slsqp", name="fit_results_phys"):
    """
    Guarda los resultados del fit físico y su matriz de covarianza en formato JSON.
    Adaptado para recibir un objeto OptimizeResult de SciPy en lugar de zfit nativo.
    """
    output_folder = os.path.join(base_dir, f"{bin_n}")
    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{name}.json")
    
    params_dict = {}
    
    # Extraemos los errores de la diagonal de la matriz de covarianza
    errors_phys = np.sqrt(np.diag(cov_phys))
    
    for i, obs_name in enumerate(obs_order):
        val = phys_dict.get(obs_name, 0.0)
        sym_err = errors_phys[i]
        
        # Asumimos errores simétricos por la aproximación Hessiana
        params_dict[obs_name] = {
            'value': float(val), 
            'error': float(sym_err), 
            'error_low': float(-sym_err), 
            'error_up': float(sym_err), 
            'error_source': "hessian" 
        }

    cov_list = cov_phys.tolist()
    
    # Mapeo de los atributos de SciPy a tu estándar de zfit
    data_to_save = {
        'bin_index': str(bin_n), 
        'valid': bool(result_scipy.success),       # Equivalente a result_zfit.valid
        'converged': bool(result_scipy.success),   # Equivalente a result_zfit.converged
        'fmin': float(result_scipy.fun),           # Equivalente a result_zfit.fmin
        'status': str(result_scipy.message),       # Equivalente a result_zfit.status
        'parameters': params_dict,
        'covariance': cov_list
    }
    
    with open(output_file, 'w') as f:
        json.dump(data_to_save, f, indent=4)
        
    print(f"[CheckPoint] Resultados físicos guardados en: {output_file}")
    return output_file

def select_q2_bin(df, n_bin, cut):
    q2_bins = dict()
    q2_bins = { "bin0":[1.1,23.0],   "bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0],
                "bin4":[6.0, 7.0],   "bin5":[7.0, 8.0], "bin6": [8.0, 11.0],"bin7":[11.0, 12.5],
                "bin8":[12.5, 15.0], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
    df_ = df[(df[cut]>=q2_bins[n_bin][0]) & (df[cut] <= q2_bins[n_bin][1])].copy()
    return df_



def run_fit_slsqp_physical(pdf, data, params_list):

    nll = zfit.loss.UnbinnedNLL(model=pdf, data=data)    
    def compute_nll(x):
        zfit.param.set_values(params_list, x)
        return float(nll.value())


    constraints = [
        # 0 <= FL <= 1 
        {'type': 'ineq', 'fun': lambda x: x[0]},
        {'type': 'ineq', 'fun': lambda x: 1.0 - x[0]},
        
        # |S3| <= 1/2(1 - FL)
        {'type': 'ineq', 'fun': lambda x: 0.5 * (1.0 - x[0]) - x[1]},
        {'type': 'ineq', 'fun': lambda x: 0.5 * (1.0 - x[0]) + x[1]},
        
        # S3^2 + 4/9 AFB^2 + S9^2 <= 1/4(1 - FL)^2
        {'type': 'ineq', 'fun': lambda x: 0.25 * (1.0 - x[0])**2 - (x[1]**2 + (4.0/9.0)*x[3]**2 + x[2]**2)},
        
        # 4S4^2 + S7^2 <= FL(1 - FL - 2S3)
        {'type': 'ineq', 'fun': lambda x: x[0] * (1.0 - x[0] - 2.0*x[1]) - (4.0*x[4]**2 + x[5]**2)},
        
        # S5^2 + 4S8^2 <= FL(1 - FL + 2S3)
        {'type': 'ineq', 'fun': lambda x: x[0] * (1.0 - x[0] + 2.0*x[1]) - (x[6]**2 + 4.0*x[7]**2)}

        # # Restricción del determinante de la matriz de densidad
        # {'type': 'eq', 'fun': lambda x: (
        #     0.25 * x[0] * (1.0 - x[0])**2 
        #     - x[0] * x[1]**2 
        #     - x[0] * ((4.0/9.0) * x[3]**2 + x[2]**2) 
        #     - 0.25 * (1.0 - x[0] + 2.0 * x[1]) * (4.0 * x[4]**2 + x[5]**2) 
        #     - 0.25 * (1.0 - x[0] - 2.0 * x[1]) * (x[6]**2 + 4.0 * x[7]**2) 
        #     - ((2.0/3.0) * x[3] * (2.0 * x[4] * x[6] + 2.0 * x[5] * x[7]) - x[2] * (4.0 * x[4] * x[7] - x[6] * x[5]))
        # )}
    ]

    # lmites iniciales
    bounds = [
        (0.0, 1.0),   # FL
        (-0.5, 0.5),  # S3
        (-0.5, 0.5),  # S9
        (-0.75, 0.75),# AFB
        (-0.36, 0.36),# S4
        (-0.70, 0.70),# S7
        (-0.70, 0.70),# S5
        (-0.36, 0.36) # S8
    ]

    # inicial
    x0 = np.array([p.value().numpy() for p in params_list])
    result_scipy = optimize.minimize(compute_nll, x0, method='SLSQP', bounds=bounds, constraints=constraints,options={'maxiter': 1000, 'ftol': 1e-6, 'disp': True})
    zfit.param.set_values(params_list, result_scipy.x)

    return result_scipy, nll


def calculate_hessian_errors(nll, params_list):
    hessian_tf = nll.hessian(params=params_list)
    H = hessian_tf.numpy()
    

    V = np.linalg.inv(H)
    variances = np.diagonal(V)
    if np.any(variances < 0):
        print("      [ADVERTENCIA] Varianzas negativas detectadas. El mínimo podría no ser válido.")
        variances = np.abs(variances)
    errors = np.sqrt(variances)
    
    return errors, V
        


In [4]:
q2_bins = {"bin1": [1.1, 2.0], "bin2": [2.0, 4.0], "bin3": [4.0, 6.0], "bin4": [6.0, 7.0], "bin5": [7.0, 8.0], "bin7": [11.0, 12.5], "bin9": [15.0, 17.0], "bin10": [17.0, 23.0]}

for binN in q2_bins.keys():
    print(f"\n{'='*60}\nProcesando {binN} con rango q2: {q2_bins[binN]}\n{'='*60}")

    # ======================================================
    # CONFIGURACIÓN DEL ESPACIO 
    # ======================================================
    cos_l = zfit.Space('cos_l', limits=(-1, 1))
    cos_k = zfit.Space('cos_k', limits=(-1, 1))
    phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
    obs_ang = cos_l * cos_k * phi  

    # # Parámetros Físicos con límites 
    # FL  = zfit.Parameter(f'FL_{binN}',  0.5, lower_limit=0.0, upper_limit=1.0)
    # S3  = zfit.Parameter(f'S3_{binN}',  0.0, lower_limit=-0.5, upper_limit=0.5)
    # S9  = zfit.Parameter(f'S9_{binN}',  0.0, lower_limit=-0.5, upper_limit=0.5)
    # AFB = zfit.Parameter(f'AFB_{binN}', 0.0, lower_limit=-0.75, upper_limit=0.75)
    # S4  = zfit.Parameter(f'S4_{binN}',  0.0, lower_limit=-0.36, upper_limit=0.36)
    # S7  = zfit.Parameter(f'S7_{binN}',  0.0, lower_limit=-0.70, upper_limit=0.70)
    # S5  = zfit.Parameter(f'S5_{binN}',  0.0, lower_limit=-0.70, upper_limit=0.70)
    # S8  = zfit.Parameter(f'S8_{binN}',  0.0, lower_limit=-0.36, upper_limit=0.36)
    
    # Parámetros Físicos 
    FL  = zfit.Parameter(f'FL_{binN}',  0.5)
    S3  = zfit.Parameter(f'S3_{binN}',  0.0)
    S9  = zfit.Parameter(f'S9_{binN}',  0.0)
    AFB = zfit.Parameter(f'AFB_{binN}', 0.0)
    S4  = zfit.Parameter(f'S4_{binN}',  0.0)
    S7  = zfit.Parameter(f'S7_{binN}',  0.0)
    S5  = zfit.Parameter(f'S5_{binN}',  0.0)
    S8  = zfit.Parameter(f'S8_{binN}',  0.0)
    

    r_keys = ['FL', 'S3', 'S9', 'AFB', 'S4', 'S7', 'S5', 'S8']
    fit_params_list_phy = [FL, S3, S9, AFB, S4, S7, S5, S8]

    # ======================================================
    # CONSTRUCCIÓN DE PDFs Y CARGA DE DATOS
    # ======================================================
    pdf_ang_phy = FullAngular_Physical_PDF(obs_ang, FL, S3, S9, AFB, S4, S7, S5, S8)
    
    obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
    data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)

    # ======================================================
    # FITS
    # ======================================================
    print("\n" + "="*60)
    print(">>> INICIANDO FIT GEN LEVEL PHYSICAL PDF (SLSQP) <<<")
    print("="*60)
    result_scipy, nll = run_fit_slsqp_physical(pdf_ang_phy, data_true, fit_params_list_phy)
    print("\nValores  y Errores (Hess):")        
    errores, cov_matrix = calculate_hessian_errors(nll, fit_params_list_phy)
    for param, error in zip(fit_params_list_phy, errores):
        print(f"{param.name}: {param.value().numpy():.4f} ± {error:.4f}")

    # ======================================================
    # GUARDADO DE RESULTADO
    # ======================================================
    base_dir = "fit_results/gen_phy_slsqp"
    out_dir = f"{base_dir}/{binN}"
    os.makedirs(out_dir, exist_ok=True)        
    cov_file = f"{out_dir}/cov_matrix_{binN}.npy"
    np.save(cov_file, cov_matrix)
    
    res_file = f"{out_dir}/fit_results_{binN}.txt"
    with open(res_file, "w") as f:
        f.write("Parameter\tValue\tError\n")
        for param, error in zip(fit_params_list_phy, errores):
            f.write(f"{param.name}\t{param.value().numpy():.6f}\t{error:.6f}\n")

        phys_dict_values = {param.name.split('_')[0]: param.value().numpy() for param in fit_params_list_phy}        
        # Definir el orden estricto de los observables (obs_order)
        r_keys = ['FL', 'S3', 'S9', 'AFB', 'S4', 'S7', 'S5', 'S8']
        save_phy_results_slsqp(result_scipy=result_scipy, phys_dict=phys_dict_values, cov_phys=cov_matrix, obs_order=r_keys, bin_n=binN, base_dir="fit_results/gen_phy_slsqp",name=f"fit_results_gen_physical_slsqp_{binN}")    
        print(f"\n[INFO] Resultados guardados en: {out_dir}")

    


Procesando bin1 con rango q2: [1.1, 2.0]

>>> INICIANDO FIT GEN LEVEL PHYSICAL PDF (SLSQP) <<<
Optimization terminated successfully    (Exit mode 0)
            Current function value: 36028.56214732087
            Iterations: 19
            Function evaluations: 194
            Gradient evaluations: 19

Valores  y Errores (Hess):
FL_bin1: 0.6776 ± 0.0062
S3_bin1: 0.0010 ± 0.0084
S9_bin1: -0.0086 ± 0.0085
AFB_bin1: -0.0025 ± 0.0058
S4_bin1: 0.0291 ± 0.0102
S7_bin1: 0.0128 ± 0.0101
S5_bin1: -0.0105 ± 0.0101
S8_bin1: -0.0102 ± 0.0104
[CheckPoint] Resultados físicos guardados en: fit_results/gen_phy_slsqp/bin1/fit_results_gen_physical_slsqp_bin1.json

[INFO] Resultados guardados en: fit_results/gen_phy_slsqp/bin1

Procesando bin2 con rango q2: [2.0, 4.0]

>>> INICIANDO FIT GEN LEVEL PHYSICAL PDF (SLSQP) <<<
Optimization terminated successfully    (Exit mode 0)
            Current function value: 69867.38165470707
            Iterations: 20
            Function evaluations: 202
          